# Laboratorio: Búsqueda en entornos complejos

## Búsqueda Local 

En este laboratorio se estudiarán algoritmos de búsqueda local y heurística en un entorno de espacio de estados complejo. El problema elegido es el de las 8 reinas, en el cual se deben ubicar 8 reinas sobre un tablero de ajedrez de 8x8 de manera que ninguna reina pueda atacar a otra. 

A diferencia de los métodos de búsqueda sistemática, los algoritmos de búsqueda local no construyen explícitamente un árbol completo, sino que exploran el espacio de estados moviéndose entre configuraciones vecinas. 

## Descripción del problema

El problema de las 8 reinas consiste en colocar 8 reinas sobre un tablero de 8x8 de forma que:
- no compartan la misma fila
- no compartan la misma columna
- no compartan la misma diagonal.

*Representación del estado*

Cada estado se representará como un arreglo de longitud 8:



In [1]:
estado = [0, 4, 7, 5, 2, 6, 1, 3]

significa:
- en la columna 0 hay una reina en la fila 0,
- en la columna 1 hay una reina en la fila 4,
- ...
- en la columna 7 hay una reina en la fila 3.

## Implementación

*En parejas*, deberán implementar: 
- Hill Climbing
- Una variación del Hill Climbing (Stochastic/ First-choice/ Random-restart)
- Beam Search con beam width = _k_

Cada grupo debe realizar experimentos comparativos entre los algoritmos implementados.

### Experimentos requeridos:

Ejecutar cada algoritmo 1000 veces con estados iniciales aleatorios y reportar:

- porcentaje de éxito,
- promedio de largo de episodio (hasta éxito o estancarse)
- promedio de valor heurístico final,
- tiempo promedio de ejecución.

Para Beam Search, repetir los experimentos con distintos valores de _k_ (2,5,10). 

## Discuta 

¿Qué tan frecuentemente Hill Climbing encuentra una solución?

¿Qué tipo de problemas presenta Hill Climbing en este dominio?

¿La variante elegida mejora el desempeño? ¿Por qué?

¿Cómo afecta el valor de k en Beam Search?

¿Cuál algoritmo resultó más efectivo?

¿Qué relación existe entre costo computacional y tasa de éxito?

## Funciones útiles

In [2]:
def es_solucion(estado):
    """
    Verifica si un estado representa una solución válida al problema
    de las 8 reinas.

    Parámetros:
        estado (list): lista de 8 enteros, donde el índice representa
                       la columna y el valor representa la fila.

    Retorna:
        bool: True si es una solución válida, False en caso contrario.
    """
    if not isinstance(estado, list) or len(estado) != 8:
        return False

    # Verificar que todas las filas sean enteros entre 0 y 7
    for fila in estado:
        if not isinstance(fila, int) or fila < 0 or fila > 7:
            return False

    n = 8

    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1 = estado[c1]
            r2 = estado[c2]

            # Misma fila
            if r1 == r2:
                return False

            # Misma diagonal
            if abs(r1 - r2) == abs(c1 - c2):
                return False

    return True

print(es_solucion([0, 4, 7, 5, 2, 6, 1, 3]))  # True
print(es_solucion([0, 1, 2, 3, 4, 5, 6, 7]))  # False


def heuristica(estado):
    """
    Calcula el número de pares de reinas que se atacan entre sí.
    Un estado solución tiene heurística 0.
    """
    conflictos = 0
    n = 8

    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1 = estado[c1]
            r2 = estado[c2]

            if r1 == r2 or abs(r1 - r2) == abs(c1 - c2):
                conflictos += 1

    return conflictos

True
False


# **Implementación de Algoritmos**

## **Hill Climbing**

In [5]:
import pandas as pd
import random
import time

# El índice representa la columna, y el valor, la fila de la reina
def generarEstadoAleatorio():
    estado = []
    for i in range(8):
        numero = random.randint(0, 7)
        estado.append(numero)
    return estado

def generarVecinos(estado):
    posiblesVecinos = []

    for columna in range(8):
        filaActual = estado[columna]
        for nuevaFila in range(8):
            if nuevaFila != filaActual:
                vecino = estado[:]
                vecino[columna] = nuevaFila
                posiblesVecinos.append(vecino)

    return posiblesVecinos

# Hill Climbing
def hillClimbing():
    estadoActual = generarEstadoAleatorio()
    cantidadPasos = 0

    while True:
        heuristicaActual = heuristica(estadoActual)

        # Ya encontramos una solucion
        if heuristicaActual == 0:
            return {
                "estadoFinal": estadoActual,
                "cantidadPasos": cantidadPasos,
                "exito": True,
                "heuristicaFinal": heuristicaActual
            }

        vecinos = generarVecinos(estadoActual)

        mejorVecino = None
        mejorHeuristica = float("inf")

        for vecino in vecinos:
            heuristicaVecino = heuristica(vecino)
            if heuristicaVecino < mejorHeuristica:
                mejorHeuristica = heuristicaVecino
                mejorVecino = vecino

        # Si ya no hay mejora, nos quedamos estancados
        if mejorHeuristica >= heuristicaActual:
            return {
                "estadoFinal": estadoActual,
                "exito": False,
                "cantidadPasos": cantidadPasos,
                "heuristicaFinal": heuristicaActual
            }

        estadoActual = mejorVecino
        cantidadPasos += 1

def ejecutarExperimentosHillClimbing():
    exitos = 0
    sumaPasos = 0
    sumaHeuristicaFinal = 0
    sumaTiempo = 0.0

    for i in range(1000):
        inicio = time.perf_counter()
        resultado = hillClimbing()
        fin = time.perf_counter()

        if resultado["exito"]:
            exitos += 1

        sumaPasos += resultado["cantidadPasos"]
        sumaHeuristicaFinal += resultado["heuristicaFinal"]
        sumaTiempo += (fin - inicio)

    estadisticas = {
        "porcentajeExito": (exitos / 1000) * 100,
        "promedioLargoEpisodio": sumaPasos / 1000,
        "promedioHeuristicaFinal": sumaHeuristicaFinal / 1000,
        "tiempoPromedioSegundos": sumaTiempo / 1000
    }

    return estadisticas

estadisticasHillClimbing = ejecutarExperimentosHillClimbing()

dfHillClimbing = pd.DataFrame(
    [estadisticasHillClimbing],
    index=["Hill Climbing"]
)

dfHillClimbing

,porcentajeExito,promedioLargoEpisodio,promedioHeuristicaFinal,tiempoPromedioSegundos
Hill Climbing,14.4,3.24,1.27,0.000472
